# 05 — SQL Interview Correctness Tests (PostgreSQL 18)

80 **question-specific invariants** plus targeted fixture-coverage probes. Tests are read-only and do not alter your database. An invariant PASS is **not** a mathematical proof of full result correctness; it checks one independent property. Some questions have stronger row/count/aggregate comparisons, others only weaker plausibility constraints. Review the `strength` column before interpreting results.

Run after the 80-coverage reseed notebook. Connection settings are loaded from `db_config.ini`.


In [ ]:
import psycopg2, pandas as pd, csv, traceback
from IPython.display import display
# Import the shared database configuration regardless of whether VS Code
# starts the kernel in coding-exercises/ or coding-exercises/sql/.
from pathlib import Path
import sys
_SQL_DIR = next((p for p in (Path.cwd(), Path.cwd() / "sql")
                 if (p / "db_connection.py").is_file()), None)
if _SQL_DIR is None:
    raise FileNotFoundError(
        "Cannot locate sql/db_connection.py. Open coding-exercises/ or its sql/ folder in VS Code."
    )
if str(_SQL_DIR) not in sys.path:
    sys.path.insert(0, str(_SQL_DIR))
from db_connection import connect, connection_settings

conn = connect(autocommit=True)
with conn.cursor() as cur:
    cur.execute('SHOW server_version')
    print('PostgreSQL:', cur.fetchone()[0])


## Run 80 targeted correctness checks

Each test runs the reference solution in a CTE, then checks an independent property. **No writes** are made. The result report includes `PASS`, `FAIL`, `ERROR`, and `NO_ROWS`. `NO_ROWS` is flagged even when a boolean invariant is vacuously true.


In [ ]:
TEST_CASES = {'Q01': {'title': 'Q01 — Customer Order Summary',
         'sql': "SELECT c.customer_id, c.first_name || ' ' || c.last_name AS full_name,\n"
                '       COUNT(o.order_id) AS number_of_orders, SUM(o.total_amount) AS total_order_amount\n'
                'FROM customers c JOIN orders o ON o.customer_id=c.customer_id\n'
                'GROUP BY c.customer_id, c.first_name, c.last_name',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual a WHERE a.number_of_orders<>(SELECT COUNT(*) FROM '
                  'orders o WHERE o.customer_id=a.customer_id) OR a.total_order_amount IS DISTINCT FROM '
                  '(SELECT SUM(total_amount) FROM orders o WHERE o.customer_id=a.customer_id))',
         'fixture': None},
 'Q02': {'title': 'Q02 — Customers Without Orders',
         'sql': "SELECT c.customer_id, c.first_name || ' ' || c.last_name AS full_name, c.city, "
                'c.signup_date\n'
                'FROM customers c LEFT JOIN orders o ON o.customer_id=c.customer_id\n'
                'WHERE o.order_id IS NULL',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual a WHERE EXISTS(SELECT 1 FROM orders o WHERE '
                  'o.customer_id=a.customer_id)) AND (SELECT COUNT(*) FROM actual)=(SELECT COUNT(*) FROM '
                  'customers c WHERE NOT EXISTS(SELECT 1 FROM orders o WHERE o.customer_id=c.customer_id))',
         'fixture': 'SELECT EXISTS(SELECT 1 FROM customers c WHERE NOT EXISTS(SELECT 1 FROM orders o WHERE '
                    'o.customer_id=c.customer_id))'},
 'Q03': {'title': 'Q03 — Orders With Customer and Employee',
         'sql': "SELECT o.order_id,o.order_date,c.first_name||' '||c.last_name customer_name,\n"
                "e.first_name||' '||e.last_name employee_name,o.total_amount\n"
                'FROM orders o JOIN customers c ON c.customer_id=o.customer_id\n'
                'LEFT JOIN employees e ON e.employee_id=o.sales_employee_id',
         'check': 'SELECT (SELECT COUNT(*) FROM actual)=(SELECT COUNT(*) FROM orders) AND NOT EXISTS(SELECT '
                  '1 FROM actual WHERE customer_name IS NULL)',
         'fixture': None},
 'Q04': {'title': 'Q04 — Employees and Managers',
         'sql': "SELECT e.employee_id,e.first_name||' '||e.last_name employee_name,\n"
                "m.first_name||' '||m.last_name manager_name\n"
                'FROM employees e LEFT JOIN employees m ON m.employee_id=e.manager_id',
         'check': 'SELECT (SELECT COUNT(*) FROM actual)=(SELECT COUNT(*) FROM employees)',
         'fixture': None},
 'Q05': {'title': 'Q05 — Products Never Ordered',
         'sql': 'SELECT p.product_id,p.product_name,p.category,p.unit_price\n'
                'FROM products p LEFT JOIN order_items oi ON oi.product_id=p.product_id\n'
                'WHERE oi.order_item_id IS NULL',
         'check': 'SELECT (SELECT COUNT(*) FROM actual)=(SELECT COUNT(*) FROM products p WHERE NOT '
                  'EXISTS(SELECT 1 FROM order_items oi WHERE oi.product_id=p.product_id)) AND NOT '
                  'EXISTS(SELECT 1 FROM actual a WHERE EXISTS(SELECT 1 FROM order_items oi WHERE '
                  'oi.product_id=a.product_id))',
         'fixture': 'SELECT EXISTS(SELECT 1 FROM products p WHERE NOT EXISTS(SELECT 1 FROM order_items oi '
                    'WHERE oi.product_id=p.product_id))'},
 'Q06': {'title': 'Q06 — Customer Support Tickets',
         'sql': "SELECT t.ticket_id,c.first_name||' '||c.last_name customer_name,\n"
                "e.first_name||' '||e.last_name assigned_employee,t.status,t.priority\n"
                'FROM support_tickets t JOIN customers c ON c.customer_id=t.customer_id\n'
                'LEFT JOIN employees e ON e.employee_id=t.assigned_employee_id',
         'check': 'SELECT (SELECT COUNT(*) FROM actual)=(SELECT COUNT(*) FROM support_tickets)',
         'fixture': None},
 'Q07': {'title': 'Q07 — Order Line Details',
         'sql': 'SELECT oi.order_id,p.product_name,oi.quantity,oi.unit_price,\n'
                'oi.quantity*oi.unit_price AS line_total\n'
                'FROM order_items oi JOIN products p ON p.product_id=oi.product_id',
         'check': 'SELECT (SELECT COUNT(*) FROM actual)=(SELECT COUNT(*) FROM order_items) AND NOT '
                  'EXISTS(SELECT 1 FROM actual WHERE line_total IS DISTINCT FROM quantity*unit_price)',
         'fixture': None},
 'Q08': {'title': 'Q08 — Same-City Customers',
         'sql': "SELECT c1.customer_id,c1.first_name||' '||c1.last_name customer1,\n"
                "c2.customer_id,c2.first_name||' '||c2.last_name customer2,c1.city\n"
                'FROM customers c1 JOIN customers c2 ON c1.city=c2.city AND c1.customer_id<c2.customer_id',
         'check': 'SELECT (SELECT COUNT(*) FROM actual)=(SELECT COUNT(*) FROM customers c1 JOIN customers c2 '
                  'ON c1.city=c2.city AND c1.customer_id<c2.customer_id)',
         'fixture': 'SELECT EXISTS(SELECT 1 FROM customers a JOIN customers b ON a.city=b.city AND '
                    'a.customer_id<b.customer_id)'},
 'Q09': {'title': 'Q09 — Projects and Team Members',
         'sql': "SELECT p.project_id,p.project_name,e.employee_id,e.first_name||' '||e.last_name "
                'employee_name,\n'
                'pa.role_on_project,pa.allocation_pct\n'
                'FROM projects p LEFT JOIN project_assignments pa ON pa.project_id=p.project_id\n'
                'LEFT JOIN employees e ON e.employee_id=pa.employee_id',
         'check': 'SELECT (SELECT COUNT(*) FROM actual)=(SELECT COUNT(*) FROM projects p LEFT JOIN '
                  'project_assignments pa ON pa.project_id=p.project_id)',
         'fixture': None},
 'Q10': {'title': 'Q10 — Orders With Payment Status',
         'sql': "SELECT o.order_id,c.first_name||' '||c.last_name customer_name,p.payment_status\n"
                'FROM orders o JOIN customers c ON c.customer_id=o.customer_id\n'
                'LEFT JOIN payments p ON p.order_id=o.order_id',
         'check': 'SELECT (SELECT COUNT(*) FROM actual)=(SELECT COUNT(*) FROM orders o LEFT JOIN payments p '
                  'ON p.order_id=o.order_id)',
         'fixture': None},
 'Q11': {'title': 'Q11 — Orders by Status',
         'sql': 'SELECT status, COUNT(*) AS order_count FROM orders GROUP BY status ORDER BY order_count '
                'DESC',
         'check': 'SELECT (SELECT SUM(order_count) FROM actual)=(SELECT COUNT(*) FROM orders) AND (SELECT '
                  'COUNT(*) FROM actual)=(SELECT COUNT(DISTINCT status) FROM orders)',
         'fixture': None},
 'Q12': {'title': 'Q12 — Revenue by Customer',
         'sql': "SELECT c.customer_id,c.first_name||' '||c.last_name full_name,SUM(o.total_amount) "
                'completed_revenue\n'
                'FROM customers c JOIN orders o ON o.customer_id=c.customer_id\n'
                "WHERE o.status='Completed' GROUP BY c.customer_id,c.first_name,c.last_name\n"
                'HAVING SUM(o.total_amount)>2000 ORDER BY completed_revenue DESC',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual a WHERE completed_revenue IS DISTINCT FROM (SELECT '
                  'SUM(total_amount) FROM orders o WHERE o.customer_id=a.customer_id AND '
                  "status='Completed'))",
         'fixture': None},
 'Q13': {'title': 'Q13 — Department Salary Statistics',
         'sql': 'SELECT d.department_name,COUNT(e.employee_id) employee_count,MIN(e.salary) min_salary,\n'
                'MAX(e.salary) max_salary,ROUND(AVG(e.salary),2) avg_salary\n'
                'FROM departments d LEFT JOIN employees e ON e.department_id=d.department_id\n'
                'GROUP BY d.department_id,d.department_name',
         'check': 'SELECT (SELECT SUM(employee_count) FROM actual)=(SELECT COUNT(*) FROM employees WHERE '
                  'department_id IS NOT NULL)',
         'fixture': None},
 'Q14': {'title': 'Q14 — Monthly Revenue',
         'sql': "SELECT DATE_TRUNC('month',order_date)::date AS month,SUM(total_amount) revenue\n"
                "FROM orders WHERE status='Completed' GROUP BY 1 ORDER BY 1",
         'check': 'SELECT (SELECT SUM(revenue) FROM actual) IS NOT DISTINCT FROM (SELECT SUM(total_amount) '
                  "FROM orders WHERE status='Completed')",
         'fixture': "SELECT COUNT(DISTINCT DATE_TRUNC('month',order_date))>=2 FROM orders WHERE "
                    "status='Completed'"},
 'Q15': {'title': 'Q15 — Conditional Order Counts',
         'sql': 'SELECT c.customer_id,COUNT(o.order_id) total_orders,\n'
                "COUNT(*) FILTER (WHERE o.status='Completed') completed_orders,\n"
                "COUNT(*) FILTER (WHERE o.status='Cancelled') cancelled_orders,\n"
                "COUNT(*) FILTER (WHERE o.status='Pending') pending_orders\n"
                'FROM customers c LEFT JOIN orders o ON o.customer_id=c.customer_id GROUP BY c.customer_id',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual a WHERE a.total_orders IS DISTINCT FROM (SELECT '
                  'COUNT(*) FROM orders o WHERE o.customer_id=a.customer_id))',
         'fixture': None},
 'Q16': {'title': 'Q16 — Product Sales Performance',
         'sql': 'SELECT p.product_id,p.product_name,COALESCE(SUM(oi.quantity),0) units_sold,\n'
                'COUNT(oi.order_item_id) order_lines,COALESCE(SUM(oi.quantity*oi.unit_price),0) '
                'sales_revenue\n'
                'FROM products p LEFT JOIN order_items oi ON oi.product_id=p.product_id GROUP BY '
                'p.product_id,p.product_name',
         'check': 'SELECT (SELECT SUM(units_sold) FROM actual)=(SELECT SUM(quantity) FROM order_items)',
         'fixture': None},
 'Q17': {'title': 'Q17 — High-Volume Cities',
         'sql': 'SELECT city,COUNT(*) customer_count FROM customers GROUP BY city HAVING COUNT(*)>=3 ORDER '
                'BY customer_count DESC',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual WHERE customer_count<3)',
         'fixture': None},
 'Q18': {'title': 'Q18 — Ticket Status Summary',
         'sql': 'SELECT priority,COUNT(*) total_tickets,\n'
                "COUNT(*) FILTER (WHERE status='Open') open_tickets,\n"
                "COUNT(*) FILTER (WHERE status='Resolved') resolved_tickets,\n"
                "COUNT(*) FILTER (WHERE status='Closed') closed_tickets\n"
                'FROM support_tickets GROUP BY priority',
         'check': 'SELECT (SELECT SUM(total_tickets) FROM actual)=(SELECT COUNT(*) FROM support_tickets)',
         'fixture': None},
 'Q19': {'title': 'Q19 — Average Order Value by Segment',
         'sql': 'SELECT c.segment,COUNT(o.order_id) order_count,COALESCE(SUM(o.total_amount),0) '
                'total_revenue,\n'
                'ROUND(AVG(o.total_amount),2) average_order_value\n'
                'FROM customers c LEFT JOIN orders o ON o.customer_id=c.customer_id GROUP BY c.segment',
         'check': 'SELECT (SELECT SUM(order_count) FROM actual)=(SELECT COUNT(*) FROM orders)',
         'fixture': None},
 'Q20': {'title': 'Q20 — Department Project Allocation',
         'sql': 'SELECT d.department_name,COUNT(DISTINCT pa.employee_id) assigned_employees,\n'
                'ROUND(AVG(pa.allocation_pct),2) avg_allocation\n'
                'FROM departments d LEFT JOIN employees e ON e.department_id=d.department_id\n'
                'LEFT JOIN project_assignments pa ON pa.employee_id=e.employee_id GROUP BY '
                'd.department_id,d.department_name',
         'check': 'SELECT (SELECT COUNT(*) FROM actual)=(SELECT COUNT(*) FROM departments)',
         'fixture': None},
 'Q21': {'title': 'Q21 — Latest Order per Customer',
         'sql': 'WITH x AS (SELECT o.*,ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY order_date '
                'DESC,order_id DESC) rn FROM orders o)\n'
                'SELECT * FROM x WHERE rn=1',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual a WHERE EXISTS(SELECT 1 FROM orders o WHERE '
                  'o.customer_id=a.customer_id AND (o.order_date,o.order_id)>(a.order_date,a.order_id)))',
         'fixture': None},
 'Q22': {'title': 'Q22 — Salary Rank by Department',
         'sql': 'SELECT employee_id,department_id,first_name,last_name,salary,\n'
                'RANK() OVER(PARTITION BY department_id ORDER BY salary DESC) salary_rank FROM employees',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual a WHERE salary_rank<>(SELECT 1+COUNT(*) FROM '
                  'employees e WHERE e.department_id=a.department_id AND e.salary>a.salary))',
         'fixture': 'SELECT EXISTS(SELECT 1 FROM employees a JOIN employees b ON '
                    'a.department_id=b.department_id AND a.salary=b.salary AND '
                    'a.employee_id<>b.employee_id)'},
 'Q23': {'title': 'Q23 — Dense Salary Rank',
         'sql': 'SELECT employee_id,department_id,first_name,last_name,salary,\n'
                'DENSE_RANK() OVER(PARTITION BY department_id ORDER BY salary DESC) dense_salary_rank FROM '
                'employees',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual a WHERE dense_salary_rank<>(SELECT '
                  '1+COUNT(DISTINCT salary) FROM employees e WHERE e.department_id=a.department_id AND '
                  'e.salary>a.salary))',
         'fixture': None},
 'Q24': {'title': 'Q24 — Previous Customer Order',
         'sql': 'SELECT customer_id,order_id,order_date,total_amount,\n'
                'LAG(order_date) OVER(PARTITION BY customer_id ORDER BY order_date,order_id) '
                'previous_order_date,\n'
                'LAG(total_amount) OVER(PARTITION BY customer_id ORDER BY order_date,order_id) '
                'previous_order_amount FROM orders',
         'check': 'SELECT (SELECT COUNT(*) FROM actual)=(SELECT COUNT(*) FROM orders)',
         'fixture': None},
 'Q25': {'title': 'Q25 — Next Event Time',
         'sql': 'SELECT user_id,event_id,event_type,event_ts,\n'
                'LEAD(event_ts) OVER(PARTITION BY user_id ORDER BY event_ts,event_id) next_event_ts FROM '
                'events',
         'check': 'SELECT (SELECT COUNT(*) FROM actual)=(SELECT COUNT(*) FROM events)',
         'fixture': None},
 'Q26': {'title': 'Q26 — Running Customer Spend',
         'sql': 'SELECT customer_id,order_id,order_date,total_amount,\n'
                'SUM(total_amount) OVER(PARTITION BY customer_id ORDER BY order_date,order_id ROWS UNBOUNDED '
                'PRECEDING) running_spend FROM orders',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual a WHERE running_spend IS DISTINCT FROM (SELECT '
                  'SUM(total_amount) FROM orders o WHERE o.customer_id=a.customer_id AND '
                  '(o.order_date,o.order_id)<=(a.order_date,a.order_id)))',
         'fixture': None},
 'Q27': {'title': 'Q27 — Department Salary vs Average',
         'sql': 'SELECT employee_id,department_id,salary,\n'
                'ROUND(AVG(salary) OVER(PARTITION BY department_id),2) department_avg,\n'
                'ROUND(salary-AVG(salary) OVER(PARTITION BY department_id),2) difference FROM employees',
         'check': 'SELECT (SELECT COUNT(*) FROM actual)=(SELECT COUNT(*) FROM employees)',
         'fixture': None},
 'Q28': {'title': 'Q28 — Order Sequence Number',
         'sql': 'SELECT customer_id,order_id,order_date,\n'
                'ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY order_date,order_id) order_sequence '
                'FROM orders',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual a WHERE order_sequence<>(SELECT COUNT(*) FROM '
                  'orders o WHERE o.customer_id=a.customer_id AND '
                  '(o.order_date,o.order_id)<=(a.order_date,a.order_id)))',
         'fixture': None},
 'Q29': {'title': 'Q29 — Running Product Units',
         'sql': 'WITH d AS (SELECT oi.product_id,o.order_date,SUM(oi.quantity) units\n'
                'FROM order_items oi JOIN orders o ON o.order_id=oi.order_id GROUP BY '
                'oi.product_id,o.order_date)\n'
                'SELECT *,SUM(units) OVER(PARTITION BY product_id ORDER BY order_date ROWS UNBOUNDED '
                'PRECEDING) cumulative_units FROM d',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual WHERE cumulative_units<units)',
         'fixture': None},
 'Q30': {'title': 'Q30 — Customer Order Gap',
         'sql': 'WITH x AS (SELECT o.*,LAG(order_date) OVER(PARTITION BY customer_id ORDER BY '
                'order_date,order_id) prev_date FROM orders o)\n'
                'SELECT customer_id,order_id,order_date,prev_date,order_date-prev_date AS '
                'days_since_previous FROM x',
         'check': 'SELECT (SELECT COUNT(*) FROM actual)=(SELECT COUNT(*) FROM orders)',
         'fixture': None},
 'Q31': {'title': 'Q31 — Above-Average Salary',
         'sql': 'SELECT * FROM employees WHERE salary>(SELECT AVG(salary) FROM employees) ORDER BY salary '
                'DESC',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual WHERE salary<=(SELECT AVG(salary) FROM employees))',
         'fixture': None},
 'Q32': {'title': 'Q32 — Above Department Average',
         'sql': 'WITH a AS (SELECT department_id,AVG(salary) avg_salary FROM employees GROUP BY '
                'department_id)\n'
                'SELECT e.*,a.avg_salary FROM employees e JOIN a USING(department_id) WHERE '
                'e.salary>a.avg_salary',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual WHERE salary<=avg_salary)',
         'fixture': None},
 'Q33': {'title': 'Q33 — Customers Above Average Spend',
         'sql': 'WITH s AS (SELECT customer_id,SUM(total_amount) spend FROM orders GROUP BY customer_id)\n'
                'SELECT c.customer_id,c.first_name,c.last_name,s.spend FROM s JOIN customers c '
                'USING(customer_id)\n'
                'WHERE s.spend>(SELECT AVG(spend) FROM s) ORDER BY s.spend DESC',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual a WHERE a.spend<=(SELECT AVG(x.spend) FROM (SELECT '
                  'SUM(total_amount) spend FROM orders GROUP BY customer_id)x))',
         'fixture': None},
 'Q34': {'title': 'Q34 — Best-Selling Products',
         'sql': 'WITH u AS (SELECT p.product_id,p.product_name,COALESCE(SUM(oi.quantity),0) units\n'
                'FROM products p LEFT JOIN order_items oi USING(product_id) GROUP BY '
                'p.product_id,p.product_name)\n'
                'SELECT * FROM u WHERE units>(SELECT AVG(units) FROM u) ORDER BY units DESC',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual a WHERE a.units<=(SELECT AVG(units) FROM (SELECT '
                  'p.product_id,COALESCE(SUM(oi.quantity),0) units FROM products p LEFT JOIN order_items oi '
                  'USING(product_id) GROUP BY p.product_id)z))',
         'fixture': None},
 'Q35': {'title': 'Q35 — Orders Above Customer Average',
         'sql': 'SELECT o.* FROM orders o WHERE total_amount>(SELECT AVG(o2.total_amount) FROM orders o2 '
                'WHERE o2.customer_id=o.customer_id)',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual a WHERE total_amount<=(SELECT AVG(total_amount) '
                  'FROM orders o WHERE o.customer_id=a.customer_id))',
         'fixture': None},
 'Q36': {'title': 'Q36 — Departments With High Earners',
         'sql': 'SELECT d.* FROM departments d WHERE EXISTS\n'
                '(SELECT 1 FROM employees e WHERE e.department_id=d.department_id AND e.salary>110000)',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual d WHERE NOT EXISTS(SELECT 1 FROM employees e WHERE '
                  'e.department_id=d.department_id AND e.salary>110000))',
         'fixture': None},
 'Q37': {'title': 'Q37 — Customers With No Support Tickets',
         'sql': 'SELECT c.* FROM customers c WHERE NOT EXISTS (SELECT 1 FROM support_tickets t WHERE '
                't.customer_id=c.customer_id)',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual c WHERE EXISTS(SELECT 1 FROM support_tickets t '
                  'WHERE t.customer_id=c.customer_id))',
         'fixture': None},
 'Q38': {'title': 'Q38 — Project Staffing Pipeline',
         'sql': 'WITH s AS (SELECT project_id,COUNT(DISTINCT employee_id) headcount,SUM(allocation_pct) '
                'total_allocation\n'
                'FROM project_assignments GROUP BY project_id)\n'
                'SELECT p.project_id,p.project_name,s.headcount,s.total_allocation FROM projects p JOIN s '
                'USING(project_id) WHERE s.headcount>=3',
         'check': 'SELECT (SELECT COUNT(*) FROM actual)>0',
         'fixture': None},
 'Q39': {'title': 'Q39 — Revenue Share by Category',
         'sql': 'WITH c AS (SELECT p.category,SUM(oi.quantity*oi.unit_price) revenue FROM order_items oi '
                'JOIN products p USING(product_id) GROUP BY p.category),\n'
                't AS (SELECT SUM(revenue) total FROM c)\n'
                'SELECT c.*,ROUND(100.0*c.revenue/NULLIF(t.total,0),2) revenue_pct FROM c CROSS JOIN t ORDER '
                'BY revenue DESC',
         'check': 'SELECT ABS((SELECT SUM(revenue_pct) FROM actual)-100)<0.1',
         'fixture': None},
 'Q40': {'title': 'Q40 — Repeat Customers',
         'sql': 'WITH x AS (SELECT customer_id,COUNT(*) order_count,MIN(order_date) '
                'first_order_date,MAX(order_date) latest_order_date,SUM(total_amount) lifetime_order_value\n'
                'FROM orders GROUP BY customer_id HAVING COUNT(*)>=3)\n'
                'SELECT c.customer_id,c.first_name,c.last_name,x.* FROM x JOIN customers c '
                'USING(customer_id)',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual WHERE order_count<3)',
         'fixture': None},
 'Q41': {'title': 'Q41 — Orders in Last 30 Days of Dataset',
         'sql': 'SELECT * FROM orders WHERE order_date BETWEEN (SELECT MAX(order_date)-29 FROM orders) AND '
                '(SELECT MAX(order_date) FROM orders) ORDER BY order_date',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual WHERE order_date<(SELECT MAX(order_date)-29 FROM '
                  'orders) OR order_date>(SELECT MAX(order_date) FROM orders))',
         'fixture': None},
 'Q42': {'title': 'Q42 — Monthly Order Trend',
         'sql': "SELECT DATE_TRUNC('month',order_date)::date AS month,COUNT(*) order_count,SUM(total_amount) "
                'total_revenue FROM orders GROUP BY 1 ORDER BY 1',
         'check': 'SELECT (SELECT SUM(order_count) FROM actual)=(SELECT COUNT(*) FROM orders)',
         'fixture': None},
 'Q43': {'title': 'Q43 — Month-over-Month Revenue Change',
         'sql': "WITH m AS (SELECT DATE_TRUNC('month',order_date)::date AS month,SUM(total_amount) revenue "
                'FROM orders GROUP BY 1),\n'
                'x AS (SELECT *,LAG(revenue) OVER(ORDER BY month) previous_month_revenue FROM m)\n'
                'SELECT *,revenue-previous_month_revenue absolute_change,\n'
                'ROUND(100.0*(revenue-previous_month_revenue)/NULLIF(previous_month_revenue,0),2) '
                'percentage_change FROM x ORDER BY month',
         'check': 'SELECT (SELECT COUNT(*) FROM actual)=(SELECT COUNT(DISTINCT '
                  "DATE_TRUNC('month',order_date)) FROM orders)",
         'fixture': None},
 'Q44': {'title': 'Q44 — Latest Event per User',
         'sql': 'WITH x AS (SELECT e.*,ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY event_ts '
                'DESC,event_id DESC) rn FROM events e)\n'
                'SELECT * FROM x WHERE rn=1',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual a WHERE EXISTS(SELECT 1 FROM events e WHERE '
                  'e.user_id=a.user_id AND (e.event_ts,e.event_id)>(a.event_ts,a.event_id)))',
         'fixture': None},
 'Q45': {'title': 'Q45 — Daily Active Users',
         'sql': 'SELECT event_ts::date AS event_date,COUNT(DISTINCT user_id) daily_active_users FROM events '
                'GROUP BY 1 ORDER BY 1',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual a WHERE daily_active_users<>(SELECT COUNT(DISTINCT '
                  'user_id) FROM events e WHERE e.event_ts::date=a.event_date))',
         'fixture': None},
 'Q46': {'title': 'Q46 — Consecutive Login Days',
         'sql': 'WITH d AS (SELECT DISTINCT user_id,event_ts::date AS login_date FROM events WHERE '
                "event_type='login')\n"
                'SELECT d1.user_id,d1.login_date FROM d d1 JOIN d d2 ON d2.user_id=d1.user_id AND '
                'd2.login_date=d1.login_date-1 ORDER BY 1,2',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual a WHERE NOT EXISTS(SELECT 1 FROM events e WHERE '
                  "e.user_id=a.user_id AND e.event_type='login' AND e.event_ts::date=a.login_date-1))",
         'fixture': 'SELECT EXISTS(SELECT 1 FROM events a JOIN events b ON a.user_id=b.user_id AND '
                    "a.event_type='login' AND b.event_type='login' AND a.event_ts::date=b.event_ts::date+1)"},
 'Q47': {'title': 'Q47 — Days to Resolve Tickets',
         'sql': 'SELECT ticket_id,created_at,resolved_at,EXTRACT(EPOCH FROM(resolved_at-created_at))/86400.0 '
                'days_to_resolve\n'
                "FROM support_tickets WHERE status IN('Resolved','Closed') AND resolved_at IS NOT NULL",
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual WHERE days_to_resolve<0)',
         'fixture': None},
 'Q48': {'title': 'Q48 — Average Resolution Time by Priority',
         'sql': 'SELECT priority,ROUND(AVG(EXTRACT(EPOCH FROM(resolved_at-created_at))/3600.0),2) '
                'avg_resolution_hours\n'
                'FROM support_tickets WHERE resolved_at IS NOT NULL GROUP BY priority',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual WHERE avg_resolution_hours<0)',
         'fixture': None},
 'Q49': {'title': 'Q49 — Customers Inactive for 60 Days',
         'sql': 'SELECT c.customer_id,c.first_name,c.last_name,MAX(o.order_date) latest_order\n'
                'FROM customers c LEFT JOIN orders o USING(customer_id) GROUP BY '
                'c.customer_id,c.first_name,c.last_name\n'
                'HAVING MAX(o.order_date) IS NULL OR MAX(o.order_date)<(SELECT MAX(order_date)-60 FROM '
                'orders)',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual a WHERE latest_order IS NOT NULL AND '
                  'latest_order>=(SELECT MAX(order_date)-60 FROM orders))',
         'fixture': None},
 'Q50': {'title': 'Q50 — Quarterly Revenue',
         'sql': 'SELECT EXTRACT(YEAR FROM order_date)::int AS year,EXTRACT(QUARTER FROM order_date)::int AS '
                'quarter,\n'
                "COUNT(*) order_count,SUM(total_amount) revenue FROM orders WHERE status='Completed' GROUP "
                'BY 1,2 ORDER BY 1,2',
         'check': 'SELECT (SELECT SUM(order_count) FROM actual)=(SELECT COUNT(*) FROM orders WHERE '
                  "status='Completed')",
         'fixture': None},
 'Q51': {'title': 'Q51 — Duplicate Customer Emails',
         'sql': 'SELECT LOWER(TRIM(email)) normalized_email,COUNT(*) copies FROM customer_staging GROUP BY 1 '
                'HAVING COUNT(*)>1',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual WHERE copies<=1)',
         'fixture': 'SELECT EXISTS(SELECT 1 FROM customer_staging GROUP BY LOWER(TRIM(email)) HAVING '
                    'COUNT(*)>1)'},
 'Q52': {'title': 'Q52 — Keep Latest Customer Record',
         'sql': 'WITH x AS (SELECT *,ROW_NUMBER() OVER(PARTITION BY LOWER(TRIM(email)) ORDER BY updated_at '
                'DESC,staging_id DESC) rn FROM customer_staging)\n'
                'SELECT * FROM x WHERE rn=1',
         'check': 'SELECT (SELECT COUNT(*) FROM actual)=(SELECT COUNT(DISTINCT LOWER(TRIM(email))) FROM '
                  'customer_staging)',
         'fixture': None},
 'Q53': {'title': 'Q53 — Duplicate Events',
         'sql': 'SELECT user_id,event_type,event_ts,COUNT(*) copies FROM events GROUP BY '
                'user_id,event_type,event_ts HAVING COUNT(*)>1',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual WHERE copies<=1)',
         'fixture': 'SELECT EXISTS(SELECT 1 FROM events GROUP BY user_id,event_type,event_ts HAVING '
                    'COUNT(*)>1)'},
 'Q54': {'title': 'Q54 — Remove Duplicate Events in Result',
         'sql': 'WITH x AS (SELECT *,ROW_NUMBER() OVER(PARTITION BY user_id,event_type,event_ts ORDER BY '
                'event_id DESC) rn FROM events)\n'
                'SELECT * FROM x WHERE rn=1',
         'check': 'SELECT (SELECT COUNT(*) FROM actual)=(SELECT COUNT(*) FROM (SELECT DISTINCT '
                  'user_id,event_type,event_ts FROM events) x)',
         'fixture': None},
 'Q55': {'title': 'Q55 — Duplicate Payments',
         'sql': 'SELECT order_id,COUNT(*) payment_rows FROM payments GROUP BY order_id HAVING COUNT(*)>1',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual WHERE payment_rows<=1)',
         'fixture': 'SELECT EXISTS(SELECT 1 FROM payments GROUP BY order_id HAVING COUNT(*)>1)'},
 'Q56': {'title': 'Q56 — Latest Payment per Order',
         'sql': 'WITH x AS (SELECT *,ROW_NUMBER() OVER(PARTITION BY order_id ORDER BY payment_date '
                'DESC,payment_id DESC) rn FROM payments)\n'
                'SELECT * FROM x WHERE rn=1',
         'check': 'SELECT (SELECT COUNT(*) FROM actual)=(SELECT COUNT(DISTINCT order_id) FROM payments)',
         'fixture': None},
 'Q57': {'title': 'Q57 — Duplicate Product SKUs',
         'sql': 'SELECT UPPER(TRIM(sku)) normalized_sku,COUNT(*) copies FROM product_staging GROUP BY 1 '
                'HAVING COUNT(*)>1',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual WHERE copies<=1)',
         'fixture': 'SELECT EXISTS(SELECT 1 FROM product_staging GROUP BY UPPER(TRIM(sku)) HAVING '
                    'COUNT(*)>1)'},
 'Q58': {'title': 'Q58 — Canonical Product Staging',
         'sql': 'WITH x AS (SELECT *,ROW_NUMBER() OVER(PARTITION BY UPPER(TRIM(sku)) ORDER BY updated_at '
                'DESC,staging_id DESC) rn FROM product_staging)\n'
                'SELECT * FROM x WHERE rn=1',
         'check': 'SELECT (SELECT COUNT(*) FROM actual)=(SELECT COUNT(DISTINCT UPPER(TRIM(sku))) FROM '
                  'product_staging)',
         'fixture': None},
 'Q59': {'title': 'Q59 — Duplicate Daily Logs',
         'sql': 'SELECT pipeline_name,log_level,message,logged_at,COUNT(*) copies FROM pipeline_logs\n'
                'GROUP BY pipeline_name,log_level,message,logged_at HAVING COUNT(*)>1',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual WHERE copies<=1)',
         'fixture': 'SELECT EXISTS(SELECT 1 FROM pipeline_logs GROUP BY '
                    'pipeline_name,log_level,message,logged_at HAVING COUNT(*)>1)'},
 'Q60': {'title': 'Q60 — Deduplicated Log Feed',
         'sql': 'WITH x AS (SELECT *,ROW_NUMBER() OVER(PARTITION BY '
                'pipeline_name,log_level,message,logged_at ORDER BY log_id DESC) rn FROM pipeline_logs)\n'
                'SELECT * FROM x WHERE rn=1',
         'check': 'SELECT (SELECT COUNT(*) FROM actual)=(SELECT COUNT(*) FROM (SELECT DISTINCT '
                  'pipeline_name,log_level,message,logged_at FROM pipeline_logs)x)',
         'fixture': None},
 'Q61': {'title': 'Q61 — Top 3 Customers by Revenue',
         'sql': 'SELECT c.customer_id,c.first_name,c.last_name,SUM(o.total_amount) revenue\n'
                "FROM customers c JOIN orders o USING(customer_id) WHERE o.status='Completed'\n"
                'GROUP BY c.customer_id,c.first_name,c.last_name ORDER BY revenue DESC LIMIT 3',
         'check': 'SELECT (SELECT COUNT(*) FROM actual)<=3 AND NOT EXISTS(SELECT 1 FROM actual a WHERE '
                  'revenue IS DISTINCT FROM (SELECT SUM(total_amount) FROM orders o WHERE '
                  "o.customer_id=a.customer_id AND status='Completed'))",
         'fixture': None},
 'Q62': {'title': 'Q62 — Second-Highest Company Salary',
         'sql': 'SELECT * FROM employees WHERE salary=(SELECT MAX(salary) FROM employees WHERE '
                'salary<(SELECT MAX(salary) FROM employees))',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual WHERE salary<>(SELECT MAX(salary) FROM employees '
                  'WHERE salary<(SELECT MAX(salary) FROM employees)))',
         'fixture': None},
 'Q63': {'title': 'Q63 — Top 3 Salaries per Department',
         'sql': 'WITH x AS (SELECT e.*,DENSE_RANK() OVER(PARTITION BY department_id ORDER BY salary DESC) '
                'rnk FROM employees e)\n'
                'SELECT * FROM x WHERE rnk<=3 ORDER BY department_id,rnk',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual WHERE rnk>3)',
         'fixture': None},
 'Q64': {'title': 'Q64 — Latest Transaction per Customer',
         'sql': 'WITH x AS (SELECT t.*,ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY transaction_ts '
                'DESC,transaction_id DESC) rn FROM transactions t)\n'
                'SELECT * FROM x WHERE rn=1',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual a WHERE EXISTS(SELECT 1 FROM transactions t WHERE '
                  't.customer_id=a.customer_id AND '
                  '(t.transaction_ts,t.transaction_id)>(a.transaction_ts,a.transaction_id)))',
         'fixture': None},
 'Q65': {'title': 'Q65 — Top 2 Products per Category',
         'sql': 'WITH s AS (SELECT p.product_id,p.product_name,p.category,COALESCE(SUM(oi.quantity),0) '
                'units\n'
                'FROM products p LEFT JOIN order_items oi USING(product_id) GROUP BY '
                'p.product_id,p.product_name,p.category),\n'
                'r AS (SELECT *,DENSE_RANK() OVER(PARTITION BY category ORDER BY units DESC) rnk FROM s)\n'
                'SELECT * FROM r WHERE rnk<=2 ORDER BY category,rnk',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual WHERE rnk>3)',
         'fixture': None},
 'Q66': {'title': 'Q66 — Highest-Value Order per Month',
         'sql': "WITH x AS (SELECT o.*,DENSE_RANK() OVER(PARTITION BY DATE_TRUNC('month',order_date) ORDER "
                'BY total_amount DESC) rnk FROM orders o)\n'
                'SELECT * FROM x WHERE rnk=1',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual a WHERE EXISTS(SELECT 1 FROM orders o WHERE '
                  "DATE_TRUNC('month',o.order_date)=DATE_TRUNC('month',a.order_date) AND "
                  'o.total_amount>a.total_amount))',
         'fixture': None},
 'Q67': {'title': 'Q67 — Top 3 Cities by Revenue',
         'sql': 'SELECT c.city,SUM(o.total_amount) revenue FROM customers c JOIN orders o '
                'USING(customer_id)\n'
                'GROUP BY c.city ORDER BY revenue DESC LIMIT 3',
         'check': 'SELECT (SELECT COUNT(*) FROM actual)<=3',
         'fixture': None},
 'Q68': {'title': 'Q68 — Most Active Users',
         'sql': 'SELECT user_id,COUNT(*) event_count FROM events GROUP BY user_id ORDER BY event_count DESC '
                'LIMIT 5',
         'check': 'SELECT (SELECT COUNT(*) FROM actual)<=5 AND NOT EXISTS(SELECT 1 FROM actual a WHERE '
                  'event_count<>(SELECT COUNT(*) FROM events e WHERE e.user_id=a.user_id))',
         'fixture': None},
 'Q69': {'title': 'Q69 — Third-Latest Order per Customer',
         'sql': 'WITH x AS (SELECT o.*,ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY order_date '
                'DESC,order_id DESC) rn FROM orders o)\n'
                'SELECT * FROM x WHERE rn=3',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual a WHERE (SELECT COUNT(*) FROM orders o WHERE '
                  'o.customer_id=a.customer_id AND (o.order_date,o.order_id)>=(a.order_date,a.order_id))<>3)',
         'fixture': None},
 'Q70': {'title': 'Q70 — Top Employee by Project Load',
         'sql': 'WITH x AS (SELECT employee_id,SUM(allocation_pct) total_allocation FROM project_assignments '
                'GROUP BY employee_id),\n'
                'r AS (SELECT *,DENSE_RANK() OVER(ORDER BY total_allocation DESC) rnk FROM x)\n'
                'SELECT e.employee_id,e.first_name,e.last_name,r.total_allocation FROM r JOIN employees e '
                'USING(employee_id) WHERE r.rnk=1',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual a WHERE a.total_allocation < (SELECT '
                  'MAX(total_allocation) FROM (SELECT SUM(allocation_pct) total_allocation FROM '
                  'project_assignments GROUP BY employee_id)x))',
         'fixture': None},
 'Q71': {'title': 'Q71 — Customer 360 Snapshot',
         'sql': 'WITH o AS (SELECT customer_id,COUNT(*) lifetime_orders,SUM(total_amount) '
                'lifetime_revenue,MAX(order_date) latest_order_date FROM orders GROUP BY customer_id),\n'
                "t AS (SELECT customer_id,COUNT(*) FILTER (WHERE status='Open') open_ticket_count FROM "
                'support_tickets GROUP BY customer_id)\n'
                'SELECT c.customer_id,c.signup_date,c.segment,COALESCE(o.lifetime_orders,0) '
                'lifetime_orders,COALESCE(o.lifetime_revenue,0) lifetime_revenue,\n'
                'o.latest_order_date,COALESCE(t.open_ticket_count,0) open_ticket_count FROM customers c LEFT '
                'JOIN o USING(customer_id) LEFT JOIN t USING(customer_id)',
         'check': 'SELECT NOT EXISTS (SELECT 1 FROM actual a WHERE a.lifetime_orders <> (SELECT COUNT(*) FROM orders o WHERE o.customer_id = a.customer_id))',
         'fixture': None},
 'Q72': {'title': 'Q72 — Potentially Stuck Pipelines',
         'sql': 'WITH x AS (SELECT pipeline_name,logged_at::date AS run_date,COUNT(*) FILTER (WHERE '
                "log_level='ERROR') error_count,\n"
                'EXTRACT(EPOCH FROM(MAX(logged_at)-MIN(logged_at)))/60 duration_minutes FROM pipeline_logs '
                'GROUP BY pipeline_name,logged_at::date)\n'
                'SELECT * FROM x WHERE error_count>0 OR duration_minutes>90 ORDER BY run_date,pipeline_name',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual WHERE error_count<0 OR duration_minutes<0)',
         'fixture': None},
 'Q73': {'title': 'Q73 — Checkout Conversion Funnel',
         'sql': "SELECT COUNT(DISTINCT user_id) FILTER (WHERE event_type='product_view') viewed_users,\n"
                "COUNT(DISTINCT user_id) FILTER (WHERE event_type='add_to_cart') cart_users,\n"
                "COUNT(DISTINCT user_id) FILTER (WHERE event_type='purchase') purchased_users,\n"
                "ROUND(100.0*COUNT(DISTINCT user_id) FILTER (WHERE event_type='purchase')/\n"
                "NULLIF(COUNT(DISTINCT user_id) FILTER (WHERE event_type='product_view'),0),2) "
                'view_to_purchase_pct FROM events',
         'check': 'SELECT (SELECT viewed_users FROM actual)=(SELECT COUNT(DISTINCT user_id) FROM events '
                  "WHERE event_type='product_view')",
         'fixture': None},
 'Q74': {'title': 'Q74 — Revenue Reconciliation',
         'sql': 'SELECT o.order_id,o.total_amount,SUM(oi.quantity*oi.unit_price) item_total,\n'
                'o.total_amount-SUM(oi.quantity*oi.unit_price) difference\n'
                'FROM orders o JOIN order_items oi USING(order_id) GROUP BY o.order_id,o.total_amount\n'
                'HAVING ABS(o.total_amount-SUM(oi.quantity*oi.unit_price))>0.01',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual WHERE ABS(difference)<=0.01)',
         'fixture': 'SELECT EXISTS(SELECT 1 FROM orders o JOIN order_items oi USING(order_id) GROUP BY '
                    'o.order_id,o.total_amount HAVING '
                    'ABS(o.total_amount-SUM(oi.quantity*oi.unit_price))>0.01)'},
 'Q75': {'title': 'Q75 — Customer Churn Candidates',
         'sql': 'SELECT c.customer_id,c.first_name,c.last_name,COUNT(o.order_id) '
                'historical_orders,MAX(o.order_date) latest_order\n'
                'FROM customers c JOIN orders o USING(customer_id) GROUP BY '
                'c.customer_id,c.first_name,c.last_name\n'
                'HAVING COUNT(o.order_id)>=2 AND MAX(o.order_date)<(SELECT MAX(order_date)-90 FROM orders)',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual WHERE historical_orders<2 OR latest_order>=(SELECT '
                  'MAX(order_date)-60 FROM orders))',
         'fixture': None},
 'Q76': {'title': 'Q76 — Payment Failure Rate',
         'sql': "SELECT DATE_TRUNC('month',payment_date)::date AS month,COUNT(*) total_attempts,\n"
                "COUNT(*) FILTER (WHERE payment_status='Failed') failed_payments,\n"
                "ROUND(100.0*COUNT(*) FILTER (WHERE payment_status='Failed')/COUNT(*),2) failure_rate_pct\n"
                'FROM payments GROUP BY 1 ORDER BY 1',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual WHERE failure_rate_pct<0 OR failure_rate_pct>100 '
                  'OR failed_payments>total_attempts)',
         'fixture': None},
 'Q77': {'title': 'Q77 — Support SLA Breaches',
         'sql': 'SELECT ticket_id,priority,created_at,resolved_at,\n'
                'ROUND(EXTRACT(EPOCH FROM(resolved_at-created_at))/3600.0,2) resolution_hours,\n'
                "CASE priority WHEN 'High' THEN 24 WHEN 'Medium' THEN 48 ELSE 72 END sla_hours\n"
                'FROM support_tickets WHERE resolved_at IS NOT NULL\n'
                'AND EXTRACT(EPOCH FROM(resolved_at-created_at))/3600.0 >\n'
                "CASE priority WHEN 'High' THEN 24 WHEN 'Medium' THEN 48 ELSE 72 END",
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual WHERE resolution_hours<=sla_hours)',
         'fixture': 'SELECT EXISTS(SELECT 1 FROM support_tickets WHERE resolved_at IS NOT NULL AND '
                    "EXTRACT(EPOCH FROM(resolved_at-created_at))/3600.0 > CASE priority WHEN 'High' THEN 24 "
                    "WHEN 'Medium' THEN 48 ELSE 72 END)"},
 'Q78': {'title': 'Q78 — Pipeline Error Trend',
         'sql': 'SELECT pipeline_name,logged_at::date AS log_date,COUNT(*) total_logs,\n'
                "COUNT(*) FILTER (WHERE log_level='ERROR') error_logs,\n"
                "ROUND(100.0*COUNT(*) FILTER (WHERE log_level='ERROR')/COUNT(*),2) error_rate_pct\n"
                'FROM pipeline_logs GROUP BY pipeline_name,logged_at::date ORDER BY log_date,pipeline_name',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual WHERE error_rate_pct<0 OR error_rate_pct>100 OR '
                  'error_logs>total_logs)',
         'fixture': None},
 'Q79': {'title': 'Q79 — Product Reorder Candidates',
         'sql': 'WITH mx AS (SELECT MAX(order_date) max_date FROM orders),\n'
                's AS (SELECT oi.product_id,SUM(oi.quantity) units_30d FROM order_items oi JOIN orders o '
                'USING(order_id),mx\n'
                'WHERE o.order_date BETWEEN mx.max_date-29 AND mx.max_date GROUP BY oi.product_id)\n'
                'SELECT p.product_id,p.product_name,p.stock_quantity,p.reorder_level,COALESCE(s.units_30d,0) '
                'units_sold_last_30d\n'
                'FROM products p LEFT JOIN s USING(product_id) WHERE p.active AND '
                'p.stock_quantity<p.reorder_level',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual a JOIN products p USING(product_id) WHERE '
                  'a.stock_quantity>=a.reorder_level OR NOT p.active)',
         'fixture': 'SELECT EXISTS(SELECT 1 FROM products WHERE active AND stock_quantity<reorder_level)'},
 'Q80': {'title': 'Q80 — Suspicious Transaction Pattern',
         'sql': 'SELECT customer_id,transaction_ts::date AS transaction_date,COUNT(*) '
                'transaction_count,SUM(amount) daily_amount\n'
                'FROM transactions GROUP BY customer_id,transaction_ts::date HAVING COUNT(*)>=3 AND '
                'SUM(amount)>1000 ORDER BY daily_amount DESC',
         'check': 'SELECT NOT EXISTS(SELECT 1 FROM actual WHERE transaction_count<3 OR daily_amount<=1000)',
         'fixture': 'SELECT EXISTS(SELECT 1 FROM transactions GROUP BY customer_id,transaction_ts::date '
                    'HAVING COUNT(*)>=3 AND SUM(amount)>1000)'}}


In [ ]:
results=[]
for qid, case in TEST_CASES.items():
    sql = case['sql'].strip().rstrip(';')
    row = dict(question=qid, title=case['title'], status='ERROR', row_count=None,
               invariant_pass=None, fixture_pass=None, detail='')
    try:
        with conn.cursor() as cur:
            cur.execute('SELECT COUNT(*) FROM ('+sql+') AS actual')
            row['row_count'] = cur.fetchone()[0]
            cur.execute('WITH actual AS ('+sql+') '+case['check'])
            row['invariant_pass'] = bool(cur.fetchone()[0])
            if case['fixture']:
                cur.execute(case['fixture'])
                row['fixture_pass'] = bool(cur.fetchone()[0])
        row['status'] = ('NO_ROWS' if row['row_count']==0 else
                         'FAIL' if row['invariant_pass'] is not True or row['fixture_pass'] is False
                         else 'PASS')
    except Exception as exc:
        row['detail'] = str(exc).splitlines()[0][:350]
    results.append(row)
report = pd.DataFrame(results)
display(report)
print('STATUS COUNTS:', report.status.value_counts().to_dict())
report.to_csv('sql_interview_correctness_report.csv',index=False)
print('Saved: sql_interview_correctness_report.csv')


## Investigate failures

Inspect the exact SQL and invariant for any failing question. Some `FAIL` results may indicate a defect in the *test assumption*, not necessarily in the solution; for example, NULLs, ties, or rounding rules can require refinement.


In [ ]:
problems=report[report.status!='PASS']
display(problems)
for qid in problems.question:
    case=TEST_CASES[qid]
    print('\n',qid,case['title'])
    print('SOLUTION:\n',case['sql'])
    print('INVARIANT:\n',case['check'])
    if case['fixture']:print('FIXTURE:\n',case['fixture'])


## What this does and does not prove

- **Stronger checks:** independent source counts, row-level comparisons, group totals, and alternative anti-join logic.
- **Weaker checks:** range bounds, positive values, and non-empty checks. These cannot establish that every expected row is present.
- **Fixture probes:** verify certain edge-case scenarios exist in the seeded data; they do not inject new rows or test every possible edge case.
- **For full correctness:** compare entire result multisets against independently specified oracles (`EXCEPT ALL` both directions), plus controlled transaction-rollback fixtures. This notebook is a substantial second validation layer, **not 80 complete formal proofs**.
